# AutoPipe — flujo tabular de punta a punta

Este notebook recorre el camino completo para datos **tabulares**: de la tabla cruda
al pipeline de preprocesamiento ganador, con el espacio de búsqueda canónico
versionado que usa el experimento de meta-aprendizaje del TFG.

Es el equivalente tabular de los tres notebooks de imágenes. Se ejecuta **sin datos
externos**: el dataset se genera aquí mismo con una semilla fija.

Lo que se muestra, en orden:

1. Un dataset tabular con columnas numéricas, una categórica y valores faltantes.
2. Por qué el cargador devuelve una **tabla** y no una matriz numérica.
3. El espacio de búsqueda canónico `tabular-v1` y por qué está versionado.
4. La búsqueda en malla sobre las 144 combinaciones.
5. La garantía anti-fuga, demostrada en vivo.
6. Reproducibilidad con Optuna.
7. El reporte.

## 1. Un dataset tabular

Tres columnas numéricas de escalas muy distintas, una categórica correlacionada de
forma imperfecta con la clase, y valores faltantes. Tres clases balanceadas.

Ese perfil no es decorativo: obliga a que el escalado, la codificación y la
imputación tengan algo que hacer, que es justo lo que el espacio de búsqueda explora.
Las clases se solapan a propósito — con clases perfectamente separables cualquier
pipeline acierta y no habría nada que elegir.

In [ ]:
import logging

import numpy as np
import optuna
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier

from autopipe import (
    TABULAR_SPACE_VERSION,
    GridSearchEngine,
    OptunaSearchEngine,
    ReportGenerator,
    SelectFeatures,
    TabularDataLoader,
    build_tabular_search_space,
    describe_tabular_search_space,
)

rng = np.random.default_rng(7)
N = 150
clase = np.tile([0, 1, 2], N // 3)

tabla = pd.DataFrame({
    'medida':        clase * 0.8 + rng.normal(0, 1.2, N),      # informativa, con solape
    'ruido':         rng.normal(0, 1.0, N),                    # no informativa
    'escala_grande': clase * 120.0 + rng.normal(0, 220.0, N),  # informativa, otra escala
    'region':        np.where(rng.random(N) < 0.5,
                              clase,
                              rng.integers(0, 3, N)),          # categórica imperfecta
    'clase':         clase,
})
tabla['region'] = tabla['region'].map({0: 'norte', 1: 'sur', 2: 'este'})
tabla.loc[rng.choice(N, 12, replace=False), 'ruido'] = np.nan   # faltantes

print(tabla.dtypes.to_string())
tabla.head()

## 2. El cargador devuelve una tabla, no una matriz

`TabularDataLoader` separa la etiqueta, la codifica a enteros y **conserva los tipos
de columna**.

Esa decisión gobierna todo lo que viene. Si el cargador devolviera una matriz
numérica, las transformaciones no podrían distinguir solas las columnas categóricas
de las numéricas y habría que decírselo por parámetro. Y en cuanto un parámetro
nombra columnas, la configuración deja de ser transferible entre datasets — que es
exactamente lo que el arranque en caliente necesita hacer.

In [ ]:
cargador = TabularDataLoader(tabla, target_column='clase')
features, labels = cargador.load()

print('numéricas   :', cargador.numeric_columns_)
print('categóricas :', cargador.categorical_columns_)
print('clases      :', cargador.classes_)
print('faltantes   :', int(features.isna().sum().sum()))
print('forma       :', features.shape)

## 3. El espacio de búsqueda canónico y versionado

El espacio tabular no se arma a mano: se pide. Es **uno solo, el mismo para todos los
datasets**, y lleva un identificador de versión.

Las dos propiedades son requisitos del arranque en caliente. El recomendador
transfiere configuraciones halladas en unos datasets hacia otro que nunca vio: si
cada dataset tuviera su espacio, no habría nada que transferir.

In [ ]:
print('versión del espacio:', TABULAR_SPACE_VERSION)

pd.DataFrame(describe_tabular_search_space())

**Por qué el versionado no es burocracia.** Lo que Optuna registra por ensayo son
*índices enteros* sobre la lista de opciones de cada paso, más un índice extra que
significa «omitir este paso». Un índice no significa nada por sí mismo: depende del
orden de la lista.

Si el espacio cambiara de claves o de orden, una configuración inyectada como
arranque en caliente designaría transformaciones distintas de las previstas **sin
levantar ningún error**: la búsqueda correría, daría un número, y ese número no
mediría lo que se cree. `validate_space_version` convierte ese fallo mudo en un
rechazo explícito, y una prueba congela el índice de cada opción.

**El orden también está congelado**: `imputation → scaling → encoding → selection`.
El escalado va antes de la codificación para que actúe solo sobre las columnas
numéricas originales y deje las indicadoras *one-hot* en `0`/`1`. Estandarizarlas
llevaría una categoría rara a valores cercanos a ±10 y, con vecinos más cercanos y
distancia euclídea, esa sola columna dominaría la métrica.

Con la opción de omitir cada paso, el espacio tiene `3 × 4 × 3 × 4 = 144` pipelines.

## 4. Búsqueda en malla

El modelo base es el que fija el pre-registro del experimento: `KNeighborsClassifier`
con `k=5` y fuerza bruta. Se eligió porque opera sobre distancias, así que el
preprocesamiento le afecta de verdad; un modelo insensible produciría un efecto piso
y la comparación del experimento quedaría vacía por construcción.

In [ ]:
# El motor avisa por cada candidato inevaluable. Silenciamos ese aviso para que la
# salida quede legible y los contamos más abajo, que es más informativo.
logging.getLogger('autopipe').setLevel(logging.ERROR)

motor = GridSearchEngine(
    build_tabular_search_space(), val_size=0.3, random_state=42, task='classification'
)
mejor, metricas, resultados = motor.run(
    features, labels, KNeighborsClassifier(n_neighbors=5, algorithm='brute')
)

print('ganador:', mejor)
print()
for nombre, valor in metricas.items():
    print(f'  {nombre:>18}: {valor:.4f}')

### Los candidatos que no se pueden evaluar

El motor prueba también **omitir** cada paso. Algunos de esos candidatos son
inevaluables: sin codificación, una tabla con columnas categóricas no se puede
convertir en matriz numérica; sin imputación, quedan nulos que el modelo no acepta.

`evaluate_pipeline_safe` los registra con métricas `NaN` y la búsqueda continúa, así
que **nunca ganan**. No son errores del programa: son candidatos descartados.

In [ ]:
objetivos = np.array([registro['objective'] for registro in resultados])
invalidos = int(np.isnan(objetivos).sum())

print(f'candidatos evaluados : {len(objetivos)}')
print(f'inevaluables (NaN)   : {invalidos}  ({100 * invalidos / len(objetivos):.0f} %)')
print()
print(f'mejor macro-F1       : {np.nanmax(objetivos):.4f}')
print(f'mediana              : {np.nanmedian(objetivos):.4f}')
print(f'peor macro-F1 válido : {np.nanmin(objetivos):.4f}')
print(f'diferencia mejor-peor: {np.nanmax(objetivos) - np.nanmin(objetivos):.4f}')

**Ese número es la razón de ser de todo el proyecto.** Sobre el mismo dataset, el
mismo modelo y la misma partición, elegir bien o mal el preprocesamiento mueve el
macro-F1 cerca de medio punto. Si no lo moviera, no habría nada que optimizar ni que
recomendar, y el experimento de meta-aprendizaje quedaría vacío por construcción.

Es también la razón por la que el pre-registro fija un modelo base sensible al
preprocesamiento y criba los datasets del corpus por sensibilidad.

La fracción de inevaluables depende del dataset y llega al 56 % en tablas con
columnas categóricas *y* faltantes, como esta. Está medida y documentada en
`docs/experimentos/ESPACIO-tabular-v1.md`, porque tiene consecuencias sobre el
protocolo experimental.

## 5. La garantía anti-fuga, en vivo

El pipeline se ajusta **solo con el subconjunto de entrenamiento**. Con un paso
supervisado como la selección de características, esa garantía deja de ser sutil: una
selección que viera la validación no filtraría una media, elegiría **otras columnas**,
y el modelo entrenaría con una matriz distinta.

Abajo, la misma selección ajustada de las dos maneras sobre una tabla donde la
columna informativa cambia entre las dos mitades.

In [ ]:
import warnings

# En esta tabla diminuta hay columnas constantes dentro de una clase, así que
# f_classif avisa de ello. Es exactamente el caso que queremos mostrar.
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

trampa = pd.DataFrame({
    'a': [0.0, 0.0, 10.0, 10.0,  10.0, 10.0, 0.0, 0.0],   # separa en entrenamiento
    'b': [1.0, 1.0,  1.0,  1.0,   0.0,  0.0, 5.0, 5.0],   # separa en validación
})
etiquetas = np.array([0, 0, 1, 1, 0, 0, 1, 1])
train_idx = np.arange(0, 4)

honesto = SelectFeatures(0.5, 'f_classif').fit(trampa.iloc[train_idx], etiquetas[train_idx])
con_fuga = SelectFeatures(0.5, 'f_classif').fit(trampa, etiquetas)

print('ajustada solo con entrenamiento ->', honesto.selected_columns_)
print('ajustada con el dataset completo ->', con_fuga.selected_columns_)

Eligen columnas distintas. Por eso `evaluate_pipeline` clona el pipeline y lo ajusta
únicamente con `train_idx`, y por eso hay pruebas escritas para **fallar** si alguien
revierte ese comportamiento (`tests/test_tabular_protocol.py`).

Se verificó saboteando la función a propósito: con la fuga introducida, siete de las
once pruebas de ese archivo caen.

## 6. Reproducibilidad con Optuna

`OptunaSearchEngine` recorre el mismo espacio con un muestreador TPE. Es el motor
donde se integrará el arranque en caliente, inyectando configuraciones recomendadas
con `enqueue_trial`.

El criterio de éxito de ingeniería del anteproyecto exige que la misma semilla
produzca el mismo ganador.

> **Cuidado al leer los números.** `random_state` gobierna el muestreador *y* la
> partición de validación, así que dos semillas distintas evalúan sobre conjuntos
> distintos. Los valores de macro-F1 de abajo **no son comparables entre semillas**,
> ni con el de la búsqueda en malla de la sección 4, que usó la semilla 42. Lo único
> que la comparación entre semillas dice es cuánta variación introduce la partición
> — que es justamente por lo que el protocolo del TFG repite cada condición con tres
> semillas y toma la mediana.

In [ ]:
# Optuna registra una línea por ensayo; con 75 ensayos eso tapa el resultado.
optuna.logging.set_verbosity(optuna.logging.WARNING)


def buscar(semilla):
    motor = OptunaSearchEngine(
        build_tabular_search_space(), n_trials=25, val_size=0.3,
        random_state=semilla, task='classification',
    )
    ganador, metricas, _ = motor.run(
        features, labels, KNeighborsClassifier(n_neighbors=5, algorithm='brute')
    )
    return ganador, metricas['macro_f1']


primero, f1_primero = buscar(101)
segundo, f1_segundo = buscar(101)
distinta, f1_distinta = buscar(202)

print(f'semilla 101 (1a vez): {f1_primero:.4f}')
print(f'semilla 101 (2a vez): {f1_segundo:.4f}')
print(f'semilla 202         : {f1_distinta:.4f}')
print()
print('misma semilla, mismo ganador:', primero.get_params() == segundo.get_params())

## 7. Reporte

`ReportGenerator` es agnóstico a la tarea: deriva las columnas de métricas de los
propios resultados. El PDF es opcional y requiere el binario `wkhtmltopdf` en el
PATH, así que aquí se genera solo el HTML.

In [ ]:
reporte = ReportGenerator(output_dir='reports/tabular').generate(
    resultados, mejor, metricas, metric='macro_f1', pdf=False,
)

print('HTML  :', reporte['html'])
print('config:', reporte['config'])

---

## Para seguir

| Tema | Dónde |
|---|---|
| Cómo funciona por dentro y qué garantiza | `docs/GUIA-tecnica.md` |
| Composición congelada del espacio y su tabla de índices | `docs/experimentos/ESPACIO-tabular-v1.md` |
| Por qué el contrato tiene `fit`/`transform` | `docs/decisiones/ADR-001-contrato-fit-transform.md` |
| Protocolo experimental, fijado antes de ejecutar | `docs/experimentos/Preregistro.tex` |
| El mismo flujo en 13 instrucciones | `examples/tabular_quickstart.py` |

**Un pipeline reconstruido desde la configuración viene sin ajustar**: el ajuste
depende de los datos, no de la configuración. Con datos tabulares hay que volver a
llamar a `fit` antes de transformar.